In [1]:
!pip install -q transformers datasets accelerate

import pandas as pd
import re
import torch
import math
from transformers import pipeline
from tqdm import tqdm

In [2]:
# 1. Load Dataset dari GitHub
url_katalog = "https://raw.githubusercontent.com/danangtric/PJJ_Data_Analytics_2026/refs/heads/main/3.%20Katalog%20Pelatihan%202018-2024%20(puspa).csv"
url_skj = "https://raw.githubusercontent.com/danangtric/PJJ_Data_Analytics_2026/refs/heads/main/2.%20SKJ%20JF%20Perluasan.csv"

# Membaca data
df_katalog = pd.read_csv(url_katalog, sep=';').fillna("")
df_skj = pd.read_csv(url_skj, sep=';').fillna("")

# --- PENAMBAHAN ID DI AWAL ---
# Membuat ID otomatis: PUSPA-0001, PUSPA-0002, dst.
df_katalog.insert(0, 'id_pelatihan', [f"PUSPA-{i+1:04d}" for i in range(len(df_katalog))])
# ------------------------------

print(f"✅ Data Katalog Puspa Berhasil Dimuat dengan ID Unik.")
print(f"Total Data: {len(df_katalog)} baris")


✅ Data Katalog Puspa Berhasil Dimuat dengan ID Unik.
Total Data: 509 baris


In [3]:
df_katalog.head()

,id_pelatihan,nama_pelatihan,penyelenggara,tahun
0,PUSPA-0001,Diklat Fungsional Pemeriksa Dasar,Pusat Pendidikan dan Pelatihan Pajak,2018
1,PUSPA-0002,Diklat Teknis Substantif Spesialisasi Pemeriks...,Pusat Pendidikan dan Pelatihan Pajak,2018
2,PUSPA-0003,DTS Perpajakan Menengah,Pusat Pendidikan dan Pelatihan Pajak,2018
3,PUSPA-0004,DTSS Account Representative Dasar,Pusat Pendidikan dan Pelatihan Pajak,2018
4,PUSPA-0005,DTSS Analis IDLP Dasar,Pusat Pendidikan dan Pelatihan Pajak,2018


In [4]:
df_skj.head()

,id_skj,nama_skj
0,skj1,Advokasi Kebijakan Perpajakan
1,skj2,Analisis Data
2,skj3,Analisis Hasil Pengawasan
3,skj4,Analisis Hukum Internasional
4,skj5,Analisis Kebijakan Publik di Bidang Keuangan N...


In [5]:
import re

# 1. Ambil Label SKJ yang sudah bersih (langsung dari kolom)
candidate_labels = df_skj['nama_skj'].astype(str).tolist()
candidate_labels = [l for l in candidate_labels if l.strip() != ""]


In [6]:
# 1. Pastikan kolom nama_pelatihan ada dan bersih dari NaN
df_katalog['nama_pelatihan'] = df_katalog['nama_pelatihan'].fillna("Tanpa Nama").astype(str)

# 2. Fungsi Pembersih (Stopwords)
def penggal_kata(teks):
    if not teks: return set()
    # Kecilkan huruf dan ambil karakter a-z
    kata = re.sub(r'[^a-z\s]', ' ', str(teks).lower()).split()

    # Daftar Stopwords Lengkap (Sesuai List Anda)
    stop_words_katalog = {
        'pelatihan', 'jarak', 'jauh', 'e-learning', 'elearning', 'diklat', 'df',
        'dtsd', 'dts', 'dtss', 'training', 'excecutive', 'training', 'sharing', 'session', 'fgd',
        'focus', 'group', 'discussion', 'in', 'house', 'kemenkeu', 'corporate',
        'university', 'corpu', 'open', 'class', 'lokakarya', 'microlearning',
        'blended', 'pjj', 'seminar', 'workshop', 'uji', 'kompetensi', 'teknis',
        'of', 'trainers', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'dalam',
        'yang', 'level', 'tingkat', 'nasional', 'angkatan',
        'tahun', 'batch', 'program', 'studi', 'bimtek', 'sosialisasi'
    }
    return {k for k in kata if k not in stop_words_katalog and len(k) > 2}

# 3. Tampilkan Perbandingan Hasil Stopword
# Kita buat kolom sementara 'judul_bersih' untuk pengecekan visual
df_katalog['judul_bersih'] = df_katalog['nama_pelatihan'].apply(lambda x: " ".join(penggal_kata(x)))

print("🔍 HASIL PEMBERSIHAN STOPWORDS (Katalog Puspa):")
display(df_katalog[['nama_pelatihan', 'judul_bersih']].head(15))

🔍 HASIL PEMBERSIHAN STOPWORDS (Katalog Puspa):


,nama_pelatihan,judul_bersih
0,Diklat Fungsional Pemeriksa Dasar,fungsional dasar pemeriksa
1,Diklat Teknis Substantif Spesialisasi Pemeriks...,spesialisasi substantif pemeriksa pajak daerah
2,DTS Perpajakan Menengah,menengah perpajakan
3,DTSS Account Representative Dasar,dasar representative account
4,DTSS Analis IDLP Dasar,analis dasar idlp
5,DTSS Forensik Digital Perpajakan,perpajakan forensik digital
6,DTSS Forensik Digital Perpajakan Menengah,perpajakan forensik menengah digital
7,DTSS Juru Sita Pajak,pajak sita juru
8,DTSS Juru Sita Pajak Lanjutan,lanjutan pajak sita juru
9,DTSS Jurusita Pajak Daerah Kerja Sama Program ...,pusdiklat kota pemerintah padang kerja dengan ...


In [9]:
# 1. Inisialisasi Model Zero-Shot Classification (Otomatis deteksi GPU)
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification",
                      model="moritzlaurer/mDeBERTa-v3-base-mnli-xnli",
                      device=device)

# 2. Fungsi Mapping Cerdas
def mapping_akurat(nama_asli):
    if not nama_asli or str(nama_asli).strip() == "":
        return []

    # AI memproses judul asli
    res = classifier(nama_asli,
                     candidate_labels,
                     hypothesis_template="Pelatihan ini berkaitan dengan kompetensi {}.",
                     multi_label=True)

    tokens_katalog = penggal_kata(nama_asli)
    hasil_sementara = []

    for label, skor_ai in zip(res['labels'], res['scores']):
        tokens_skj = set(re.sub(r'[^a-z\s]', ' ', label.lower()).split())
        irisan = tokens_katalog.intersection(tokens_skj)

        # BONUS SKOR: Jika kata kunci di judul cocok dengan SKJ, kita beri dorongan kuat
        # Kita naikkan bonusnya jadi 0.3 supaya kata yang 'match' beneran naik ke atas
        bonus = len(irisan) * 0.3
        skor_akhir = min(skor_ai + bonus, 1.0)

        # FILTER KETAT: Hanya ambil jika skor akhir > 0.6 (biar gak asal tebak)
        if skor_akhir > 0.60:
            hasil_sementara.append({
                'kompetensi': label,
                'skor': skor_akhir
            })

    # URUTKAN berdasarkan skor tertinggi
    hasil_terurut = sorted(hasil_sementara, key=lambda x: x['skor'], reverse=True)

    # AMBIL MAKSIMAL 3 (tapi bisa kurang jika yang lain skornya rendah)
    return hasil_terurut[:3]

print("✅ Step 3 Berhasil Diperbarui: Sekarang lebih ketat dan maksimal 3 kompetensi.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: moritzlaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Step 3 Berhasil Diperbarui: Sekarang lebih ketat dan maksimal 3 kompetensi.


In [11]:
# 1. Pastikan kolom nama_pelatihan siap
df_katalog['nama_pelatihan'] = df_katalog['nama_pelatihan'].fillna("").astype(str)

# 2. Proses Pemetaan dengan Logika Baru (Max 3 & Skor > 0.6)
print("🚀 Menjalankan Mapping ulang dengan standar akurasi lebih tinggi...")
tqdm.pandas(desc="Mapping Progress")

# AI akan memproses ulang 509 data Anda
df_katalog['mapping_raw'] = df_katalog['nama_pelatihan'].progress_apply(mapping_akurat)

# 3. Merapikan hasil ke kolom final
def gabung_hasil(x):
    if not x: return "Perlu Review Manual" # Jika skor di bawah 0.6 semua
    return " ; ".join([item['kompetensi'] for item in x])

df_katalog['kompetensi_final'] = df_katalog['mapping_raw'].apply(gabung_hasil)

# 4. Simpan ke Excel dengan ID di awal
# Pastikan kolom-kolom ini ada di dataframe Anda
kolom_final = ['id_pelatihan', 'nama_pelatihan', 'kompetensi_final']
df_katalog[kolom_final].to_excel("Katalog_Puspa_Optimasi_Final.xlsx", index=False)

print("\n✅ PROSES SELESAI!")
print("📂 File 'Katalog_Puspa_Optimasi_Final.xlsx' sudah siap di folder Files.")

🚀 Menjalankan Mapping ulang dengan standar akurasi lebih tinggi...


Mapping Progress: 100%|██████████| 509/509 [07:23<00:00,  1.15it/s]


✅ PROSES SELESAI!
📂 File 'Katalog_Puspa_Optimasi_Final.xlsx' sudah siap di folder Files.
